In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from typing import Dict, List, Tuple
from training.data_loading import *
from training.loss_funcs import *

print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU              : {torch.cuda.get_device_name(0)}") # I have a 3090
    torch.set_float32_matmul_precision("medium")
    print("float32 matmul precision set to 'medium'")

PyTorch version  : 2.6.0+cu124
CUDA available   : True
GPU              : NVIDIA GeForce RTX 3090
float32 matmul precision set to 'medium'


In [2]:
weather_cols_all = ['temperature_2m',
       'apparent_temperature', 'dew_point_2m', 'relative_humidity_2m',
       'precipitation', 'rain', 'snowfall', 'cloud_cover', 'cloud_cover_low',
       'cloud_cover_mid', 'cloud_cover_high', 'surface_pressure',
       'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m',
       'shortwave_radiation', 'diffuse_radiation', 'direct_normal_irradiance']

other_cols = [ # these are not static
    'dam_price', 'buy_bm_price', 'sell_bm_price',
    'max_power', 'max_solar', 'max_ev'
]

cat_columns = [
    'eic_code', 'dso_desc', 'station_type', 'oblast',
    'Month', 'Day', 'Hour', 'day_of_week', 'season'
]

static_cols = [
    'latitude', 'longitude', 'eic_code', 'dso_desc', 'station_type', 'oblast'
]

calendar_cols = [
    'Month', 'Day', 'Hour', 'day_of_week', 'season'
]

time_cols = ['datetime', 'time_idx']

FUTURE_REALS = weather_cols_all + calendar_cols + static_cols + other_cols
print(f"y col is: {Y_COL}, group col is: {GROUP_COL}\n"
      f"features: {FUTURE_REALS}")

y col is: sum_of_kWh, group col is: eic_code
features: ['temperature_2m', 'apparent_temperature', 'dew_point_2m', 'relative_humidity_2m', 'precipitation', 'rain', 'snowfall', 'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high', 'surface_pressure', 'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m', 'shortwave_radiation', 'diffuse_radiation', 'direct_normal_irradiance', 'Month', 'Day', 'Hour', 'day_of_week', 'season', 'latitude', 'longitude', 'eic_code', 'dso_desc', 'station_type', 'oblast', 'dam_price', 'buy_bm_price', 'sell_bm_price', 'max_power', 'max_solar', 'max_ev']


In [3]:
# this data has a data column and categorical columns are kept intact and will need to be handled.
# "time_idx" is already built in train, val and test and is continuous through them
print("Loading train …")
train = load_and_prepare(TRAIN_PATH_WITH_DATETME)

print("Loading val   …")
val = load_and_prepare(VAL_PATH_WITH_DATETME)

print("Loading test  …")
test = load_and_prepare(TEST_PATH_WITH_DATETME)

print(f"train: {train.shape}")
print(f"val : {val.shape}")
print(f"test: {test.shape}")

Loading train …
Loading val   …
Loading test  …
train: (4586151, 38)
val : (293880, 38)
test: (295430, 38)


In [4]:
# # If a model cant handle categorical columns natively, or through embedings use this data
# # It has no datetime column and all columns are numeric, as all cat column were ohe
# # GROUP_COL is the only exception, and is not ohe. Ohe it before training
# print("Loading train …")
# train = load_and_prepare(TRAIN_PATH_OHE)
#
# print("Loading val   …")
# val = load_and_prepare(VAL_PATH_OHE)
#
# print("Loading test  …")
# test = load_and_prepare(TEST_PATH_OHE)
#
# print(f"train: {train.shape}")
# print(f"val  : {val.shape}")
# print(f"test : {test.shape}")

In [5]:
# this cell samples locations, I'll use it if training takes too long, otherwise don't touch it

TARGET_STATIONS = 395

station_stats = (
    train.groupby(GROUP_COL)
    .agg(rows=(Y_COL, "count"))
    .reset_index()
    .sort_values("rows", ascending=False)
)

sampled_stations = station_stats.sample(
    n=TARGET_STATIONS, random_state=42
)[GROUP_COL].values

print(f"Stations: {len(sampled_stations)}")

train = train[train[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)
val   = val[val[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)
test  = test[test[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)

print(f"Train rows : {len(train):,}")
print(f"Val rows   : {len(val):,}")
print(f"Test rows  : {len(test):,}")

Stations: 395
Train rows : 4,586,151
Val rows   : 293,880
Test rows  : 295,430


In [6]:
training_cutoff = train["time_idx"].max()
val_cutoff      = val["time_idx"].max()
test_cutoff     = test["time_idx"].max()

print(f"training cutoff : {training_cutoff}")
print(f"val cutoff      : {val_cutoff}")
print(f"test cutoff     : {test_cutoff}")

training cutoff : 13125
val cutoff      : 13869
test cutoff     : 14617


## Chronos fine-tuning

We fine-tune `amazon/chronos-t5-small` (T5-based, ~46M params) on the per-station hourly
consumption sequences. Chronos is **univariate by design**: the encoder only sees a
tokenized version of the target series.

### Important Chronos-specific decisions

1. **Univariate sequences**: each `eic_code` becomes one continuous sequence of
   `sum_of_kWh` values. Sliding windows of (`CONTEXT_LEN`, `PREDICTION_LEN`) are drawn
   per series.

2. **Tokenization & scaling**: we use `MeanScaleUniformBins` from the `chronos`
   package — the official Chronos tokenizer. It performs per-context **mean scaling**
   (each window is divided by `mean(|x|)`) and then uniformly bins values into
   integer token IDs. This is the canonical Chronos preprocessing — no additional
   per-series normalization is needed because mean scaling is built into the
   tokenizer and is reversed at inference via `output_transform`.

3. **Exogenous features (weather, prices, calendar, static)**: **excluded**.
   Chronos T5 / Chronos-Bolt do **not** natively support future covariates — see
   the official model card on HuggingFace (`amazon/chronos-2` is the first
   variant with covariate support). The recommended way to use covariates with
   Chronos / Chronos-Bolt is via an external covariate regressor in AutoGluon,
   which is out of scope for this notebook. Encoding tabular features into the
   token sequence would corrupt the pretrained value distribution. Prices are
   still loaded for **evaluation** (the `money_pct` loss needs them) but never
   fed to the model.

4. **Loss function**: Chronos is trained with **categorical cross-entropy** over
   tokenized value bins (it's a seq2seq language model on quantized values).
   The custom `money_pct` loss is **non-differentiable through the categorical
   token output of Chronos** — backpropagating an asymmetric financial loss
   through the discrete token distribution would require Gumbel-softmax /
   REINFORCE-style estimators that are not part of the Chronos training
   recipe. We therefore: 
   - **train** with the native CE loss (the loss returned by
     `T5ForConditionalGeneration` automatically),
   - **evaluate / select / report** with `money_pct` (and the other metrics).
   This is the honest, non-hallucinated approach.

5. **Recursive forecasting**: Chronos-T5 is trained with `prediction_length=64` by
   default. Our val/test windows are ~1 month (~720 hours), so we **roll the
   model forward** in chunks of `PREDICTION_LEN` hours (using each station's
   actual history as context — i.e. teacher-forced rolling rather than
   pure autoregressive — which matches a realistic day-ahead operational
   setting where yesterday's actuals are known when forecasting today).

In [7]:
# One-time install (skip if already installed). The official package is `chronos-forecasting`.
# It installs the `chronos` Python module which exposes ChronosPipeline, ChronosConfig,
# ChronosTokenizer and the MeanScaleUniformBins tokenizer.
# !pip install -q --upgrade chronos-forecasting transformers accelerate mlflow tqdm

In [8]:
# --- Chronos / HF imports ----------------------------------------------------
import os
import math
import json
import random
import tempfile
from pathlib import Path
from dataclasses import dataclass

import mlflow
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import autocast, GradScaler

from transformers import AutoModelForSeq2SeqLM, AutoConfig

# Official Chronos package: https://github.com/amazon-science/chronos-forecasting
from chronos import (
    ChronosConfig,
    ChronosPipeline,         # used for inference
    MeanScaleUniformBins,    # the official Chronos tokenizer
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


In [9]:
# --- Hyperparameters ---------------------------------------------------------
# `chronos-t5-small` is a good 3090 fit. Bump to `-base` if VRAM allows.
MODEL_ID         = "amazon/chronos-t5-small"

CONTEXT_LEN      = 512    # default Chronos context length
PREDICTION_LEN   = 64     # default Chronos prediction length (also used as inference chunk)

BATCH_SIZE       = 32
GRAD_ACCUM       = 1
MAX_STEPS        = 5000
WARMUP_STEPS     = 200
LR               = 1e-4
WEIGHT_DECAY     = 0.0
GRAD_CLIP        = 1.0

VAL_EVERY_STEPS  = 500
USE_BF16         = True   # 3090 supports bf16 (Ampere)
SEED             = 42

# how many context windows to draw per station per epoch (with replacement)
WINDOWS_PER_STATION = 32

INFERENCE_NUM_SAMPLES = 20  # Chronos is probabilistic; we average samples to a point forecast

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [10]:
# --- Per-station sequence extraction -----------------------------------------
# Chronos wants 1-D tensors per series. We sort by time_idx within each station and
# pull the raw target. Since validation/test are appended in time, we use ONLY the
# train portion for training windows. For rolling inference we will use
# (train + history-up-to-now) as the conditioning context.

def build_series_dict(df: pd.DataFrame) -> Dict[str, np.ndarray]:
    """{eic_code -> 1-D float32 array of sum_of_kWh, sorted by time_idx}"""
    out = {}
    for grp, gdf in df.sort_values("time_idx").groupby(GROUP_COL, sort=False):
        out[grp] = gdf[Y_COL].to_numpy(dtype=np.float32)
    return out

train_series = build_series_dict(train)
val_series   = build_series_dict(val)
test_series  = build_series_dict(test)

# Sanity
any_code = next(iter(train_series))
print(f"#stations train: {len(train_series)}")
print(f"example station '{any_code}' lengths -> train={len(train_series[any_code])}, "
      f"val={len(val_series.get(any_code, []))}, test={len(test_series.get(any_code, []))}")

# Drop stations whose train sequence is shorter than CONTEXT_LEN+PREDICTION_LEN
MIN_LEN = CONTEXT_LEN + PREDICTION_LEN
train_series = {k: v for k, v in train_series.items() if len(v) >= MIN_LEN}
print(f"#stations usable for training (len >= {MIN_LEN}): {len(train_series)}")

#stations train: 395
example station '62Z0008583037334' lengths -> train=13126, val=744, test=745
#stations usable for training (len >= 576): 395


In [11]:
# --- PyTorch Dataset (raw windows only) + GPU-side batched tokenization -----
# IMPORTANT performance note:
# The first version of this cell did Chronos tokenization inside __getitem__,
# which meant the tokenizer's torch ops (bucketize, mean-scaling) ran on CPU,
# once per sample, in a small worker pool. That made the dataloader the
# bottleneck and left the GPU at ~0% utilization.
#
# Fix: workers now produce only raw float32 windows (a couple of slices from
# pre-loaded numpy arrays — essentially free). We tokenize the whole batch on
# the GPU inside the training loop using `tokenize_batch` below. The Chronos
# tokenizer's `context_input_transform` accepts a 2-D tensor [B, T] and is
# fully vectorized, so this is the right granularity to call it at.

class ChronosWindowDataset(Dataset):
    """Random sliding-window sampler over multiple univariate series.
    Returns RAW windows; tokenization happens later on GPU."""

    def __init__(
        self,
        series: Dict[str, np.ndarray],
        context_len: int,
        prediction_len: int,
        windows_per_station: int,
    ):
        self.context_len = context_len
        self.prediction_len = prediction_len

        # Pre-build an index: list of (station_id, allowed_max_start)
        self.index = []
        for code, arr in series.items():
            max_start = len(arr) - (context_len + prediction_len)
            if max_start < 0:
                continue
            for _ in range(windows_per_station):
                self.index.append((code, max_start))
        self.series = series

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i):
        code, max_start = self.index[i]
        start = np.random.randint(0, max_start + 1)
        arr = self.series[code]
        full = arr[start : start + self.context_len + self.prediction_len]
        return torch.from_numpy(full)  # float32, shape [context_len + prediction_len]

def collate(batch):
    return torch.stack(batch, dim=0)


def move_tokenizer_to_device(tokenizer, device):
    """MeanScaleUniformBins keeps `centers` and `boundaries` as CPU tensors.
    Moving them isn't enough — `_append_eos_token` ALSO allocates fresh CPU
    tensors via torch.full(...) and torch.concats them, which throws
    "Expected all tensors to be on the same device". We therefore also
    monkey-patch `_append_eos_token` to allocate EOS / mask tensors on the
    same device as the input. This is the minimal, surgical fix; nothing
    about the tokenization logic itself changes."""
    tokenizer.centers = tokenizer.centers.to(device)
    tokenizer.boundaries = tokenizer.boundaries.to(device)

    eos_id = tokenizer.config.eos_token_id

    def _append_eos_token_on_device(token_ids, attention_mask):
        b = token_ids.shape[0]
        dev = token_ids.device
        eos_tokens = torch.full((b, 1), fill_value=eos_id, dtype=token_ids.dtype, device=dev)
        eos_mask   = torch.full((b, 1), fill_value=True,   dtype=attention_mask.dtype, device=dev)
        token_ids      = torch.concat((token_ids,      eos_tokens), dim=1)
        attention_mask = torch.concat((attention_mask, eos_mask),   dim=1)
        return token_ids, attention_mask

    tokenizer._append_eos_token = _append_eos_token_on_device
    return tokenizer


@torch.no_grad()
def tokenize_batch(
    raw: torch.Tensor,           # [B, context_len + prediction_len], on DEVICE, float32
    tokenizer: MeanScaleUniformBins,
    context_len: int,
    prediction_len: int,
) -> Dict[str, torch.Tensor]:
    ctx   = raw[:, :context_len]
    label = raw[:, context_len : context_len + prediction_len]

    # Vectorized Chronos tokenization on GPU.
    input_ids, attention_mask, scale = tokenizer.context_input_transform(ctx)
    labels, labels_mask = tokenizer.label_input_transform(label, scale)

    # HF -100 ignore_index convention
    labels = labels.masked_fill(labels_mask == 0, -100)
    return {
        "input_ids":      input_ids,
        "attention_mask": attention_mask,
        "labels":         labels,
    }

In [12]:
# --- money_pct evaluation wrapper -------------------------------------------
# Used during validation to track the *business* metric while we train with CE.
# Note: this is EVAL-ONLY. As discussed in the markdown above, we cannot use
# money_pct as the training loss for Chronos because the model emits categorical
# token IDs (not differentiable real-valued forecasts). The native CE loss is
# what shapes the predicted token distribution; money_pct is what we *select* on.

def _prices(df):
    return df["dam_price"].values, df["sell_bm_price"].values, df["buy_bm_price"].values

def money_pct_on_eval(eval_df: pd.DataFrame) -> float:
    return money_pct(eval_df[Y_COL], eval_df["pred"], *_prices(eval_df))

In [13]:
# --- Load pretrained Chronos T5 + matching tokenizer -------------------------
# The model card (amazon/chronos-t5-small) ships a `chronos_config` block in its
# HF config.json. We load both the seq2seq weights AND the ChronosConfig so the
# tokenizer's vocab/binning matches exactly what was pretrained.

torch.backends.cudnn.benchmark = True  # speeds up T5's fixed-shape kernels

hf_cfg = AutoConfig.from_pretrained(MODEL_ID)
chronos_cfg = ChronosConfig(**hf_cfg.chronos_config)

# MeanScaleUniformBins is the standard Chronos tokenizer; it does:
#   1. mean scaling: x' = x / mean(|x|)
#   2. uniform binning of x' in [low_limit, high_limit] -> token IDs
tokenizer = chronos_cfg.create_tokenizer()
assert isinstance(tokenizer, MeanScaleUniformBins), type(tokenizer)
# Move tokenizer's `centers` and `boundaries` to GPU so batched tokenization
# inside the training loop runs on-device (otherwise torch.bucketize errors).
tokenizer = move_tokenizer_to_device(tokenizer, DEVICE)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)
model.to(DEVICE)

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Loaded {MODEL_ID}  ({n_params:.1f}M params)")
print(f"context_length={chronos_cfg.context_length}, "
      f"prediction_length={chronos_cfg.prediction_length}, "
      f"n_tokens={chronos_cfg.n_tokens}")

Loaded amazon/chronos-t5-small  (46.2M params)
context_length=512, prediction_length=64, n_tokens=4096


In [14]:
# --- Build dataloaders -------------------------------------------------------
# Workers only do raw numpy slicing -> very cheap. We keep num_workers=0
# (synchronous loading in the main process) for now — multi-worker DataLoaders
# in Jupyter on Linux can deadlock when CUDA has already been initialized in
# the parent (the default `fork` start method clones an inconsistent CUDA
# context into the child). A pure-numpy slice in __getitem__ is fast enough
# that 0 workers will not be the bottleneck. If you later want workers, set
# `multiprocessing_context='spawn'` and `persistent_workers=True`, AND make
# sure no CUDA tensor is touched before the DataLoader is built.
train_ds = ChronosWindowDataset(
    series=train_series,
    context_len=CONTEXT_LEN,
    prediction_len=PREDICTION_LEN,
    windows_per_station=WINDOWS_PER_STATION,
)
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate,
    pin_memory=True,
    drop_last=True,
)
print(f"Train windows / epoch : {len(train_ds):,}")
print(f"Steps    / epoch       : {len(train_loader):,}")

Train windows / epoch : 12,640
Steps    / epoch       : 395


In [15]:
# --- Preflight: prove every piece works in isolation BEFORE the training loop -
# If you saw "progress bar appears but doesn't move and CPU+GPU are both 0%",
# this cell will tell you exactly where it stalls. Each step prints BEFORE
# starting and AFTER finishing, with timing. If a print appears without its
# matching "ok" line for several seconds, that's your culprit.

import time, sys

def _t(label):
    print(f"[preflight] {label} ...", flush=True)
    return time.time()

def _ok(label, t0):
    print(f"[preflight] {label} ok  ({time.time()-t0:.2f}s)", flush=True)

# 1. dataset slicing
t0 = _t("1/6 dataset __getitem__")
sample = train_ds[0]
print(f"            sample shape={tuple(sample.shape)} dtype={sample.dtype} "
      f"min={float(sample.min()):.3f} max={float(sample.max()):.3f}", flush=True)
_ok("1/6", t0)

# 2. one batch through the dataloader
t0 = _t("2/6 dataloader -> single batch")
_iter = iter(train_loader)
raw_cpu = next(_iter)
print(f"            batch shape={tuple(raw_cpu.shape)} dtype={raw_cpu.dtype}", flush=True)
_ok("2/6", t0)

# 3. host -> GPU copy (this is also where weird CUDA states show up)
t0 = _t("3/6 batch.to(DEVICE)")
raw_gpu = raw_cpu.to(DEVICE, non_blocking=False)  # blocking=True for clearer timing
torch.cuda.synchronize()
print(f"            on device={raw_gpu.device}", flush=True)
_ok("3/6", t0)

# 4. GPU-side Chronos tokenization
t0 = _t("4/6 tokenize_batch on GPU")
_batch = tokenize_batch(raw_gpu, tokenizer, CONTEXT_LEN, PREDICTION_LEN)
torch.cuda.synchronize()
for k, v in _batch.items():
    print(f"            {k}: shape={tuple(v.shape)} dtype={v.dtype} device={v.device}", flush=True)
_ok("4/6", t0)

# 5. one full forward pass under autocast
t0 = _t("5/6 model forward (bf16 autocast)")
amp_dtype_pf = torch.bfloat16 if USE_BF16 else torch.float16
model.train()
with autocast("cuda", dtype=amp_dtype_pf):
    _out = model(
        input_ids=_batch["input_ids"],
        attention_mask=_batch["attention_mask"],
        labels=_batch["labels"],
    )
torch.cuda.synchronize()
print(f"            loss={_out.loss.item():.4f}", flush=True)
_ok("5/6", t0)

# 6. one backward + optimizer step
t0 = _t("6/6 backward + optimizer step")
_opt = AdamW(model.parameters(), lr=1e-5)  # throwaway, just to test the step
_out.loss.backward()
_opt.step()
_opt.zero_grad(set_to_none=True)
torch.cuda.synchronize()
_ok("6/6", t0)

# 7. memory snapshot
print(f"[preflight] CUDA mem allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB "
      f"reserved: {torch.cuda.memory_reserved()/1e9:.2f} GB", flush=True)
print("[preflight] ALL GREEN — proceed to the training loop cell.", flush=True)


[preflight] 1/6 dataset __getitem__ ...
            sample shape=(576,) dtype=torch.float32 min=11.898 max=28.672
[preflight] 1/6 ok  (0.00s)
[preflight] 2/6 dataloader -> single batch ...
            batch shape=(32, 576) dtype=torch.float32
[preflight] 2/6 ok  (0.02s)
[preflight] 3/6 batch.to(DEVICE) ...
            on device=cuda:0
[preflight] 3/6 ok  (0.00s)
[preflight] 4/6 tokenize_batch on GPU ...
            input_ids: shape=(32, 513) dtype=torch.int64 device=cuda:0
            attention_mask: shape=(32, 513) dtype=torch.bool device=cuda:0
            labels: shape=(32, 65) dtype=torch.int64 device=cuda:0
[preflight] 4/6 ok  (0.11s)
[preflight] 5/6 model forward (bf16 autocast) ...


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


            loss=2.3823
[preflight] 5/6 ok  (1.41s)
[preflight] 6/6 backward + optimizer step ...
[preflight] 6/6 ok  (0.32s)
[preflight] CUDA mem allocated: 0.85 GB reserved: 8.94 GB
[preflight] ALL GREEN — proceed to the training loop cell.


In [16]:
# --- Rolling inference helpers ----------------------------------------------
# Chronos is trained with prediction_length=PREDICTION_LEN. To cover the full
# val / test horizon (~720h), we predict in chunks: predict next PREDICTION_LEN,
# advance the context window by PREDICTION_LEN using the *true* observed values
# (teacher-forced rolling — realistic operational setting), repeat.
#
# We instantiate a ChronosPipeline that wraps our (currently training) model so
# we get the full predict() machinery (tokenize -> generate -> output_transform).
#
# IMPORTANT: we must NOT cast the training model to bf16. Doing so would put
# the optimizer's fp32 state out of sync with the now-bf16 params and crash
# AdamW's `_foreach_lerp_`. Autocast handles bf16 matmul during training; for
# inference we keep the params in their training dtype (fp32) and rely on
# autocast inside `pipeline.predict` (which we call ourselves below).

def make_pipeline(model_to_wrap) -> ChronosPipeline:
    """Wrap our in-memory fine-tuned T5 in a ChronosPipeline for inference.

    Does NOT change the dtype/device of the wrapped model — leaves it exactly
    as the training loop set it. The pipeline's tokenizer is independent of
    the GPU-patched one we use for training, so we leave it as-is.
    """
    pipe = ChronosPipeline.from_pretrained(
        MODEL_ID,
        device_map=DEVICE,
        torch_dtype=torch.float32,   # match the training model dtype
    )
    # Replace the pipeline's underlying model with our (fine-tuned) one.
    # ChronosPipeline holds the model under .model.model (ChronosModel wraps T5).
    pipe.model.model = model_to_wrap
    pipe.model.model.eval()
    return pipe


@torch.no_grad()
def rolling_predict(
    pipeline: ChronosPipeline,
    history: np.ndarray,        # shape (T_hist,)  past actuals available before forecast start
    horizon_actuals: np.ndarray,  # shape (H,)     actual values over the eval horizon
    chunk: int = PREDICTION_LEN,
    num_samples: int = INFERENCE_NUM_SAMPLES,
) -> np.ndarray:
    """Roll a Chronos point-forecast across `H` hours in chunks of `chunk`.

    For each chunk we use the true past (history + previously revealed actuals)
    as context, predict `chunk` steps, take the median across samples as the
    point forecast, then advance the window by `chunk` real observed values.
    """
    H = len(horizon_actuals)
    preds = np.empty(H, dtype=np.float32)
    cursor = 0
    full_past = list(history.astype(np.float32))

    while cursor < H:
        steps = min(chunk, H - cursor)
        ctx = torch.tensor(full_past[-CONTEXT_LEN:], dtype=torch.float32)
        # Run pipeline.predict under autocast for bf16 speed without mutating
        # parameter dtypes.
        with autocast("cuda", dtype=torch.bfloat16 if USE_BF16 else torch.float32):
            forecast = pipeline.predict(
                context=ctx,
                prediction_length=steps,
                num_samples=num_samples,
            )
        # forecast: [1, num_samples, steps] -> point forecast = median over samples
        point = forecast[0].float().median(dim=0).values.cpu().numpy()
        preds[cursor : cursor + steps] = point
        # advance using TRUE actuals (rolling teacher-forcing)
        full_past.extend(horizon_actuals[cursor : cursor + steps].tolist())
        cursor += steps
    return preds


def evaluate_on_split(
    pipeline: ChronosPipeline,
    eval_df: pd.DataFrame,
    history_series: Dict[str, np.ndarray],
    desc: str = "eval",
) -> pd.DataFrame:
    """Run rolling forecast for every station in eval_df and return an aligned
    DataFrame with columns: [GROUP_COL, datetime, time_idx, Y_COL, pred,
    dam_price, sell_bm_price, buy_bm_price].
    """
    out_chunks = []
    eval_sorted = eval_df.sort_values([GROUP_COL, "time_idx"])
    for code, gdf in tqdm(eval_sorted.groupby(GROUP_COL, sort=False), desc=desc):
        hist = history_series.get(code)
        if hist is None or len(hist) < CONTEXT_LEN:
            continue
        actuals = gdf[Y_COL].to_numpy(dtype=np.float32)
        preds = rolling_predict(pipeline, hist, actuals)

        out = gdf[[GROUP_COL, "datetime", "time_idx", Y_COL,
                   "dam_price", "sell_bm_price", "buy_bm_price"]].copy()
        out["pred"] = preds
        out_chunks.append(out)
    return pd.concat(out_chunks, ignore_index=True)

In [17]:
# --- Tiny in-loop val routine (for monitoring during training) ---------------
# Running rolling inference over every station every 500 steps is too slow.
# We instead sample a handful of stations and forecast a single chunk.

VAL_QUICK_STATIONS = 16
_val_codes = list(val_series.keys())
_val_codes_sample = random.sample(_val_codes, min(VAL_QUICK_STATIONS, len(_val_codes)))

@torch.no_grad()
def quick_val_money_pct(model_to_eval) -> float:
    """One-chunk-per-station rolling: cheap proxy of full money_pct."""
    pipe = make_pipeline(model_to_eval)
    rows = []
    for code in _val_codes_sample:
        hist = train_series.get(code)
        if hist is None or len(hist) < CONTEXT_LEN:
            continue
        actuals = val_series[code][:PREDICTION_LEN]
        if len(actuals) < PREDICTION_LEN:
            continue
        preds = rolling_predict(pipe, hist, actuals, chunk=PREDICTION_LEN, num_samples=10)
        # join prices for these PREDICTION_LEN hours from val
        v = val[(val[GROUP_COL] == code)].sort_values("time_idx").head(PREDICTION_LEN)
        if len(v) < PREDICTION_LEN:
            continue
        rows.append(pd.DataFrame({
            Y_COL:           actuals,
            "pred":          preds,
            "dam_price":     v["dam_price"].values,
            "sell_bm_price": v["sell_bm_price"].values,
            "buy_bm_price":  v["buy_bm_price"].values,
        }))
    if not rows:
        return float("nan")
    df = pd.concat(rows, ignore_index=True)
    return money_pct(df[Y_COL], df["pred"], *_prices(df))

In [20]:
# --- Training loop -----------------------------------------------------------
mlflow.set_experiment("chronos_t5_electricity")
mlflow.start_run(run_name=f"chronos-t5-small-ctx{CONTEXT_LEN}-pred{PREDICTION_LEN}")

mlflow.log_params({
    "model_id":              MODEL_ID,
    "context_len":           CONTEXT_LEN,
    "prediction_len":        PREDICTION_LEN,
    "batch_size":            BATCH_SIZE,
    "grad_accum":            GRAD_ACCUM,
    "max_steps":             MAX_STEPS,
    "warmup_steps":          WARMUP_STEPS,
    "lr":                    LR,
    "weight_decay":          WEIGHT_DECAY,
    "grad_clip":             GRAD_CLIP,
    "use_bf16":              USE_BF16,
    "windows_per_station":   WINDOWS_PER_STATION,
    "n_train_stations":      len(train_series),
    "n_train_rows":          len(train),
    "n_val_rows":            len(val),
    "n_test_rows":           len(test),
    "train_date_min":        str(train["datetime"].min()),
    "train_date_max":        str(train["datetime"].max()),
    "val_date_min":          str(val["datetime"].min()),
    "val_date_max":          str(val["datetime"].max()),
    "test_date_min":         str(test["datetime"].min()),
    "test_date_max":         str(test["datetime"].max()),
    "features_used":         f"univariate target only ({Y_COL})",
    "features_excluded":     "weather, prices, calendar, static, max_* — Chronos T5 has no native covariate support; prices kept for eval only",
    "loss_train":            "cross_entropy_on_value_tokens (Chronos native)",
    "loss_eval":             "money_pct (selection metric) + money, MAE, RMSE, MAPE, SMAPE",
    "inference_num_samples": INFERENCE_NUM_SAMPLES,
})

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def lr_lambda(step):
    if step < WARMUP_STEPS:
        return step / max(1, WARMUP_STEPS)
    progress = (step - WARMUP_STEPS) / max(1, MAX_STEPS - WARMUP_STEPS)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

amp_dtype = torch.bfloat16 if USE_BF16 else torch.float16
scaler = GradScaler("cuda", enabled=(amp_dtype == torch.float16))  # bf16 doesn't need grad scaling

best_money_pct = float("inf")
best_step = -1
best_dir = Path(tempfile.mkdtemp(prefix="chronos_best_"))

model.train()
step = 0
pbar = tqdm(total=MAX_STEPS, desc="train")
print("[train] entering loop, building data_iter ...", flush=True)
data_iter = iter(train_loader)
print("[train] data_iter ready, starting first step", flush=True)
_first_step_logged = False

while step < MAX_STEPS:
    try:
        raw = next(data_iter)
    except StopIteration:
        data_iter = iter(train_loader)
        raw = next(data_iter)
    if not _first_step_logged:
        print(f"[train] step 1 got batch shape={tuple(raw.shape)}", flush=True)

    # 1. raw windows -> GPU (small fp32 transfer)
    raw = raw.to(DEVICE, non_blocking=True)
    # 2. batched Chronos tokenization on GPU
    batch = tokenize_batch(raw, tokenizer, CONTEXT_LEN, PREDICTION_LEN)
    if not _first_step_logged:
        print(f"[train] step 1 tokenized: input_ids={tuple(batch['input_ids'].shape)} "
              f"labels={tuple(batch['labels'].shape)}", flush=True)

    with autocast("cuda", dtype=amp_dtype):
        out = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"],
        )
        loss = out.loss / GRAD_ACCUM
    if not _first_step_logged:
        print("[train] step 1 forward done", flush=True)

    if amp_dtype == torch.float16:
        scaler.scale(loss).backward()
    else:
        loss.backward()
    if not _first_step_logged:
        print("[train] step 1 backward done", flush=True)

    if (step + 1) % GRAD_ACCUM == 0:
        if amp_dtype == torch.float16:
            scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        if amp_dtype == torch.float16:
            scaler.step(optimizer); scaler.update()
        else:
            optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()
    if not _first_step_logged:
        torch.cuda.synchronize()
        print(f"[train] step 1 optimizer step done, loss={float(out.loss.item()):.4f}", flush=True)
        _first_step_logged = True

    step += 1
    pbar.update(1)
    # IMPORTANT: .item() forces a CUDA sync. Don't do it every step.
    if step % 50 == 0:
        loss_val = float(out.loss.item())
        lr_val = float(scheduler.get_last_lr()[0])
        pbar.set_postfix(loss=f"{loss_val:.4f}", lr=f"{lr_val:.2e}")
        mlflow.log_metric("train_ce_loss", loss_val, step=step)
        mlflow.log_metric("lr", lr_val, step=step)

    if step % VAL_EVERY_STEPS == 0 or step == MAX_STEPS:
        model.eval()
        v_money_pct = quick_val_money_pct(model)
        mlflow.log_metric("val_money_pct_quick", v_money_pct, step=step)
        tqdm.write(f"[step {step}] quick val money_pct = {v_money_pct:.4f}")
        if not math.isnan(v_money_pct) and v_money_pct < best_money_pct:
            best_money_pct = v_money_pct
            best_step = step
            model.save_pretrained(best_dir)
            # also stash the chronos config so the saved dir is a valid Chronos checkpoint
            (best_dir / "chronos_config.json").write_text(json.dumps(hf_cfg.chronos_config))
            tqdm.write(f"  ↳ new best, saved to {best_dir}")
        model.train()

pbar.close()
print(f"Best quick-val money_pct = {best_money_pct:.4f} at step {best_step}")
print(f"Best checkpoint: {best_dir}")

train:   0%|          | 0/5000 [00:00<?, ?it/s]

[train] entering loop, building data_iter ...
[train] data_iter ready, starting first step
[train] step 1 got batch shape=(32, 576)
[train] step 1 tokenized: input_ids=(32, 513) labels=(32, 65)
[train] step 1 forward done
[train] step 1 backward done
[train] step 1 optimizer step done, loss=2.4410
[step 500] quick val money_pct = 3.1162
  ↳ new best, saved to C:\Users\Lev\AppData\Local\Temp\chronos_best_rijz6aqq
[step 1000] quick val money_pct = 2.9776
  ↳ new best, saved to C:\Users\Lev\AppData\Local\Temp\chronos_best_rijz6aqq
[step 1500] quick val money_pct = 2.9952
[step 2000] quick val money_pct = 2.9043
  ↳ new best, saved to C:\Users\Lev\AppData\Local\Temp\chronos_best_rijz6aqq
[step 2500] quick val money_pct = 3.0242
[step 3000] quick val money_pct = 3.0398
[step 3500] quick val money_pct = 3.0110
[step 4000] quick val money_pct = 2.8789
  ↳ new best, saved to C:\Users\Lev\AppData\Local\Temp\chronos_best_rijz6aqq
[step 4500] quick val money_pct = 3.0406
[step 5000] quick val mon

In [21]:
# --- Reload the best checkpoint for full evaluation --------------------------
if best_step > 0:
    print(f"Reloading best checkpoint from {best_dir}")
    model = AutoModelForSeq2SeqLM.from_pretrained(best_dir).to(DEVICE)
model.eval()
pipeline = make_pipeline(model)

Reloading best checkpoint from C:\Users\Lev\AppData\Local\Temp\chronos_best_rijz6aqq


In [22]:
# --- Full rolling inference on val and test ----------------------------------
# History for val      = train series
# History for test     = train series + val series  (concatenated per station)

history_for_val = train_series
history_for_test = {
    code: np.concatenate([train_series.get(code, np.array([], dtype=np.float32)),
                          val_series.get(code, np.array([], dtype=np.float32))])
    for code in set(list(train_series.keys()) + list(val_series.keys()))
}

val_eval = evaluate_on_split(pipeline, val,  history_for_val,  desc="val rolling")
test_eval = evaluate_on_split(pipeline, test, history_for_test, desc="test rolling")

print(f"val_eval shape  : {val_eval.shape}")
print(f"test_eval shape : {test_eval.shape}")
val_eval.head()

val rolling:   0%|          | 0/395 [00:00<?, ?it/s]

test rolling:   0%|          | 0/395 [00:00<?, ?it/s]

val_eval shape  : (293880, 8)
test_eval shape : (295430, 8)


,eic_code,datetime,time_idx,sum_of_kWh,dam_price,sell_bm_price,buy_bm_price,pred
0,62Z0008583037334,2025-07-01 00:00:00+03:00,13126,21.0,5568.520020,0.01,5846.950195,22.042757
1,62Z0008583037334,2025-07-01 01:00:00+03:00,13127,19.0,5568.419922,0.01,5846.839844,18.034969
2,62Z0008583037334,2025-07-01 02:00:00+03:00,13128,16.0,5190.000000,0.01,5449.500000,14.952069
3,62Z0008583037334,2025-07-01 03:00:00+03:00,13129,16.0,4888.000000,0.01,5132.399902,14.027200
4,62Z0008583037334,2025-07-01 04:00:00+03:00,13130,15.0,4299.000000,0.01,4513.950195,14.027200


In [23]:
def _prices(df):
    return df["dam_price"].values, df["sell_bm_price"].values, df["buy_bm_price"].values

val_smape_v     = smape(val_eval[Y_COL], val_eval['pred'])
val_rmse_v      = rmse(val_eval[Y_COL], val_eval['pred'])
val_mape_v      = mape(val_eval[Y_COL], val_eval['pred'])
val_money_v     = money(val_eval[Y_COL], val_eval['pred'], *_prices(val_eval))
val_money_pct_v = money_pct(val_eval[Y_COL], val_eval['pred'], *_prices(val_eval))

test_smape_v     = smape(test_eval[Y_COL], test_eval['pred'])
test_rmse_v      = rmse(test_eval[Y_COL], test_eval['pred'])
test_mape_v      = mape(test_eval[Y_COL], test_eval['pred'])
test_money_v     = money(test_eval[Y_COL], test_eval['pred'], *_prices(test_eval))
test_money_pct_v = money_pct(test_eval[Y_COL], test_eval['pred'], *_prices(test_eval))

# additional bias / MAE that the prompt asked for
val_mae_v   = float(np.mean(np.abs(val_eval[Y_COL].values  - val_eval['pred'].values)))
test_mae_v  = float(np.mean(np.abs(test_eval[Y_COL].values - test_eval['pred'].values)))
val_bias_v  = float(val_eval['pred'].sum()  - val_eval[Y_COL].sum())
test_bias_v = float(test_eval['pred'].sum() - test_eval[Y_COL].sum())

print("── Validation ──────────────────────────────────────────────")
print(f"Aligned samples : {len(val_eval):,}")
print(f"SMAPE     : {val_smape_v:.4f}")
print(f"RMSE      : {val_rmse_v:.4f}")
print(f"MAE       : {val_mae_v:.4f}")
print(f"MAPE      : {val_mape_v:.2f} %")
print(f"MONEY     : {val_money_v:.4f}")
print(f"MONEY_PCT : {val_money_pct_v:.4f}%")
print(f"BIAS (pred_sum - actual_sum) : {val_bias_v:,.2f}")

print("── Test ────────────────────────────────────────────────────")
print(f"Aligned samples : {len(test_eval):,}")
print(f"SMAPE     : {test_smape_v:.4f}")
print(f"RMSE      : {test_rmse_v:.4f}")
print(f"MAE       : {test_mae_v:.4f}")
print(f"MAPE      : {test_mape_v:.2f} %")
print(f"MONEY     : {test_money_v:.4f}")
print(f"MONEY_PCT : {test_money_pct_v:.4f}%")
print(f"BIAS (pred_sum - actual_sum) : {test_bias_v:,.2f}")

mlflow.log_metrics({
    "val_smape":      val_smape_v,
    "val_rmse":       val_rmse_v,
    "val_mae":        val_mae_v,
    "val_mape":       val_mape_v,
    "val_money":      val_money_v,
    "val_money_pct":  val_money_pct_v,
    "val_bias":       val_bias_v,
    "test_smape":     test_smape_v,
    "test_rmse":      test_rmse_v,
    "test_mae":       test_mae_v,
    "test_mape":      test_mape_v,
    "test_money":     test_money_v,
    "test_money_pct": test_money_pct_v,
    "test_bias":      test_bias_v,
    "best_val_money_pct_quick": best_money_pct,
    "best_step":      best_step,
})

# Persist predictions + best checkpoint as artifacts
art_dir = Path(tempfile.mkdtemp(prefix="chronos_artifacts_"))
val_eval.to_parquet(art_dir / "val_predictions.parquet")
test_eval.to_parquet(art_dir / "test_predictions.parquet")
mlflow.log_artifact(art_dir / "val_predictions.parquet")
mlflow.log_artifact(art_dir / "test_predictions.parquet")
if best_step > 0:
    mlflow.log_artifacts(str(best_dir), artifact_path="best_checkpoint")

mlflow.end_run()
print(f"MLflow run logged → {mlflow.get_tracking_uri()}")

── Validation ──────────────────────────────────────────────
Aligned samples : 293,880
SMAPE     : 5.7421
RMSE      : 7.1981
MAE       : 1.3872
MAPE      : 7.10 %
MONEY     : 853307.0946
MONEY_PCT : 2.9188%
BIAS (pred_sum - actual_sum) : -148,911.00
── Test ────────────────────────────────────────────────────
Aligned samples : 295,430
SMAPE     : 5.2477
RMSE      : 8.2967
MAE       : 1.4087
MAPE      : 8.66 %
MONEY     : 1090070.3280
MONEY_PCT : 3.6887%
BIAS (pred_sum - actual_sum) : -97,981.00
MLflow run logged → file:///C:/Users/Lev/Documents/GitHub/Diploma/train_models_v2/mlruns


In [ ]:
def per_station_metrics(eval_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for grp, gdf in eval_df.groupby(GROUP_COL):
        rows.append({
            GROUP_COL:    grp,
            "n":          len(gdf),
            "SMAPE":      smape(gdf[Y_COL], gdf["pred"]),
            "RMSE":       rmse (gdf[Y_COL], gdf["pred"]),
            "MAPE":       mape (gdf[Y_COL], gdf["pred"]),
            "MONEY":      money    (gdf[Y_COL], gdf["pred"], *_prices(gdf)),
            "MONEY_PCT":  money_pct(gdf[Y_COL], gdf["pred"], *_prices(gdf)),
        })
    return pd.DataFrame(rows).sort_values("SMAPE")


test_station_metrics = per_station_metrics(test_eval)

print("Top-10 best stations (test SMAPE):")
print(test_station_metrics.head(10).to_string(index=False))
print("\nBottom-10 worst stations (test SMAPE):")
print(test_station_metrics.tail(10).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


def plot_forecast(df, eic_code, start_dt=None, end_dt=None, title_prefix=""):

    df = df[df[GROUP_COL] == eic_code].sort_values("datetime")
    if df.empty:
        raise ValueError(f"No data for EiC code: {eic_code!r}")
    if start_dt is not None:
        df = df[df["datetime"] >= pd.Timestamp(start_dt)]
    if end_dt is not None:
        df = df[df["datetime"] <= pd.Timestamp(end_dt)]
    if df.empty:
        raise ValueError("No data in the specified datetime range.")

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(df["datetime"], df[Y_COL],  label="True",      linewidth=1, color="steelblue")
    ax.plot(df["datetime"], df["pred"], label="Predicted", linewidth=1, color="tomato", alpha=0.85)
    ax.set_title(
        f"{title_prefix}{eic_code}  |  MAPE={mape(df[Y_COL], df['pred']):.3f}"
        f"  MONEY_PCT={money_pct(df[Y_COL], df['pred'], *_prices(df)):.2f}"
        f"  ({df['datetime'].min().date()} \u2013 {df['datetime'].max().date()})"
    )
    ax.set_xlabel("Datetime")
    ax.set_ylabel(Y_COL)
    ax.legend()
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
    fig.autofmt_xdate(rotation=0, ha="center")
    plt.tight_layout()
    plt.show()


# Example:
# plot_forecast(val_eval,  eic_code="<code>")
# plot_forecast(test_eval, eic_code="<code>", start_dt="2025-08-25", end_dt="2025-08-26")

In [ ]:
best_station  = test_station_metrics.iloc[0][GROUP_COL]
worst_station = test_station_metrics.iloc[-1][GROUP_COL]

print(f"Best  station (SMAPE): {best_station}")
plot_forecast(test_eval, eic_code=best_station,  title_prefix="[BEST]  ")

print(f"Worst station (SMAPE): {worst_station}")
plot_forecast(test_eval, eic_code=worst_station, title_prefix="[WORST] ")

## Inference example

Quick sanity-check showing how to use the fine-tuned pipeline on a single station
to forecast the next 64 hours.

In [ ]:
example_code = next(iter(test_series))
hist = np.concatenate([train_series[example_code], val_series.get(example_code, np.array([]))])
ctx = torch.tensor(hist[-CONTEXT_LEN:], dtype=torch.float32)

forecast = pipeline.predict(
    context=ctx,
    prediction_length=PREDICTION_LEN,
    num_samples=INFERENCE_NUM_SAMPLES,
)
# shape: [1, num_samples, prediction_length]
low, median, high = np.quantile(forecast[0].cpu().numpy(), [0.1, 0.5, 0.9], axis=0)

print(f"Station {example_code}: next {PREDICTION_LEN}h forecast (median +/- 80% band):")
print(np.stack([median, low, high], axis=1)[:8])

fig, ax = plt.subplots(figsize=(14, 4))
h_idx = np.arange(-len(ctx), 0)
f_idx = np.arange(0, PREDICTION_LEN)
ax.plot(h_idx, ctx.numpy(), color="steelblue", label="history")
ax.plot(f_idx, median,      color="tomato",    label="forecast (median)")
ax.fill_between(f_idx, low, high, color="tomato", alpha=0.2, label="80% band")
ax.axvline(0, color="gray", linestyle="--")
ax.legend(); ax.set_title(f"Chronos forecast — {example_code}"); plt.tight_layout(); plt.show()